# v04_seed_mean — train

Fixed research campaign. Gold58 development OOF; no leaderboard score claim.
Private training / Internet OFF inference. No LLM or agent calls.


In [ ]:
EXPERIMENT = {'id': 'v04_seed_mean', 'suite': 'weak', 'backbone': 'swin_t', 'feature_dim': 768, 'slices': 5, 'crop': 224, 'resize': 232, 'interpolation': 'bicubic', 'aggregation': 'class_mil', 'triplet': True, 'protocols': True, 'weak_labels': True, 'finetune': False, 'ensemble': False, 'backbone_epochs': 1, 'epochs': 20, 'batch_size': 16, 'lr': 0.001, 'weight_decay': 0.01, 'expected_training_studies': 4350, 'labels_sha256': '6f704a7bdb2f894cc49445b19ba7c4378c3f548d3449e00361e10044bee40920', 'train_dependencies': ['s10_gate'], 'pipeline_sha256': '18248a9ab6aec36da80dd866e66094b4e5c17975d4ea735ce0dca0360dd47237', 'architecture': 'spatial_v1', 'representation': 'single', 'neighbors': 5, 'centers': 5, 'hidden_dim': 128, 'heads': 4, 'sequence': True, 'position': 'rope_mm', 'plane_embedding': True, 'fusion': 'gate', 'mm_unit': 3.0, 'feature_pipeline_sha256': '70e4bc4811b24a92c5d0ece041e4e4d5f921c79f507872e6d3f520ce727da0cc', 'comparison_parent': 's10_gate', 'head_seed_offsets': [0, 1, 2]}


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable,'-m','pip','install','-q','pydicom==3.0.2','python-gdcm==3.2.6'])


In [ ]:
"""Shared source embedded into each standalone experiment notebook."""

from pathlib import Path

import hashlib

import json

import math

import numpy as np

import pandas as pd

import torch

from torch import nn

PLANES = ("Sagittal", "Coronal", "Axial")

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]

SEEDS = (42, 2026, 3407)

def file_hash(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def validate_bundle(path, expected_config):
    path = Path(path).resolve()
    manifest = json.loads((path / "assets.json").read_text(encoding="utf-8"))
    if manifest["experiment"] != expected_config:
        raise ValueError("Wrong experiment bundle: attach this experiment's completed training version.")
    files = manifest["files"]
    wheels = {p.relative_to(path).as_posix() for p in (path / "wheels").glob("*.whl")}
    declared_wheels = {name for name in files if name.startswith("wheels/") and name.endswith(".whl")}
    if (not {"encoder.ts", "model.pt"}.issubset(files) or wheels != declared_wheels
            or len(wheels) != 2
            or not any(name.startswith("wheels/pydicom-3.0.2-") for name in wheels)
            or not any(name.startswith("wheels/python_gdcm-3.2.6-") for name in wheels)):
        raise ValueError("Bundle must include hashes for both models and exactly the two required wheels.")
    for name, digest in manifest["files"].items():
        target = (path / name).resolve()
        if not target.is_relative_to(path) or not target.is_file() or file_hash(target) != digest:
            raise ValueError(f"Invalid or corrupted bundle file: {name}")
    return manifest

def find_data(input_root=Path("/kaggle/input")):
    for path in (input_root / "rsna-knee-abnormality-detection",
                 input_root / "competitions/rsna-knee-abnormality-detection"):
        if (path / "test.csv").is_file():
            return path
    raise FileNotFoundError("Attach the RSNA Knee competition data.")

def find_bundle(input_root=Path("/kaggle/input")):
    paths = list(input_root.glob("*/run/bundle/assets.json"))
    paths += list(input_root.glob("notebooks/*/*/run/bundle/assets.json"))
    paths += list(input_root.glob("kernels/*/*/run/bundle/assets.json"))
    if len(paths) != 1:
        raise ValueError("Attach exactly one completed experiment training output.")
    return paths[0].parent

def make_transform(config):
    from torchvision.transforms import InterpolationMode
    from torchvision.transforms._presets import ImageClassification
    return ImageClassification(crop_size=config["crop"], resize_size=config["resize"],
                               interpolation=InterpolationMode(config["interpolation"]), antialias=True)

def ordered_slices(folder):
    import pydicom
    paths = sorted(folder.glob("*.dcm"))
    if not paths:
        raise FileNotFoundError(f"No DICOM slices in {folder}")
    headers = [pydicom.dcmread(p, stop_before_pixels=True,
               specific_tags=["ImagePositionPatient", "ImageOrientationPatient", "InstanceNumber"]) for p in paths]
    if all(hasattr(h, "ImagePositionPatient") and hasattr(h, "ImageOrientationPatient") for h in headers):
        orientation = np.asarray(headers[0].ImageOrientationPatient, dtype=float)
        normal = np.cross(orientation[:3], orientation[3:])
        if np.linalg.norm(normal) <= .9:
            raise ValueError("Invalid DICOM orientation.")
        coordinates = np.asarray([np.dot(np.asarray(h.ImagePositionPatient, dtype=float), normal) for h in headers])
    elif all(hasattr(h, "InstanceNumber") for h in headers):
        coordinates = np.asarray([float(h.InstanceNumber) for h in headers])
    else:
        raise ValueError(f"No slice position or InstanceNumber: {folder}")
    if not np.isfinite(coordinates).all():
        raise ValueError("Non-finite slice positions.")
    order = np.argsort(coordinates, kind="stable")
    coordinates = coordinates[order]
    span = np.ptp(coordinates)
    positions = (coordinates - coordinates.min()) / span if span > 0 else np.full(len(paths), .5)
    return [paths[i] for i in order], positions.astype(np.float32)

def image_tensor(path, transform):
    import pydicom
    ds = pydicom.dcmread(path)
    pixels = ds.pixel_array.astype(np.float32)
    if pixels.ndim != 2 or not np.isfinite(pixels).all():
        raise ValueError(f"Invalid MRI slice: {path}")
    pixels = pixels * float(getattr(ds, "RescaleSlope", 1)) + float(getattr(ds, "RescaleIntercept", 0))
    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        pixels = -pixels
    low, high = np.percentile(pixels, [1, 99])
    if high <= low:
        low, high = float(pixels.min()), float(pixels.max())
    if high <= low:
        raise ValueError(f"Constant MRI slice: {path}")
    pixels = np.clip((pixels - low) / (high - low), 0, 1)
    return transform(torch.from_numpy(pixels).float().unsqueeze(0).repeat(3, 1, 1))

@torch.inference_mode()
def study_tokens(uid, series, image_root, encoder, transform, config, device):
    count, dim = config["slices"], config["feature_dim"]
    features = torch.zeros(3, count, dim)
    mask = torch.zeros(3, count, dtype=torch.bool)
    positions = torch.zeros(3, count)
    rows = series.loc[series.StudyInstanceUID == uid]
    for plane_index, plane in enumerate(PLANES):
        choices = rows.loc[rows.Anatomical_Plane == plane].sort_values(
            ["Fluid_Sensitive", "Fat_Suppression", "SeriesInstanceUID"], ascending=[False, False, True])
        if choices.empty:
            continue
        paths, coords = ordered_slices(image_root / uid / choices.iloc[0].SeriesInstanceUID)
        indices = np.unique(np.rint(np.linspace(.25, .75, count) * (len(paths) - 1)).astype(int))
        batch = torch.stack([image_tensor(paths[i], transform) for i in indices]).to(device)
        output = encoder(batch).cpu().float()
        if output.shape != (len(indices), dim) or not torch.isfinite(output).all():
            raise ValueError("Unexpected encoder features.")
        features[plane_index, :len(indices)] = output
        mask[plane_index, :len(indices)] = True
        positions[plane_index, :len(indices)] = torch.from_numpy(coords[indices])
    if not mask.any():
        raise ValueError(f"No supported MRI planes for study {uid}")
    return features, mask, positions

def pooled_features(tokens, mask):
    clean = torch.where(mask[..., None], tokens, 0.)
    return (clean.sum(2) / mask.sum(2).clamp_min(1)[..., None]).flatten(1)

class StudyHead(nn.Module):
    def __init__(self, config, mean, std):
        super().__init__()
        self.mode = config["aggregation"]
        self.position = config["position"]
        self.register_buffer("mean", mean.clone())
        self.register_buffer("std", std.clone())
        if self.mode == "mean":
            self.classifier = nn.Linear(3 * config["feature_dim"], 12)
        elif self.mode == "attention":
            self.project = nn.Linear(config["feature_dim"], 64)
            layer = nn.TransformerEncoderLayer(64, 4, 128, dropout=0., batch_first=True, norm_first=True)
            self.attention = nn.TransformerEncoder(layer, 1, enable_nested_tensor=False)
            self.classifier = nn.Linear(3 * 64, 12)
            # Initialize common layers identically before adding optional plane parameters.
            self.plane_embedding = nn.Parameter(torch.zeros(3, 64)) if config["plane_embedding"] else None
            self.register_buffer("frequencies", torch.exp(torch.arange(0, 64, 2) * (-math.log(10000.) / 64)))
        else:
            raise ValueError(f"Unknown aggregation: {self.mode}")

    def forward(self, tokens, mask, positions):
        if self.mode == "mean":
            return self.classifier((pooled_features(tokens, mask) - self.mean) / self.std)
        clean = torch.where(mask[..., None], tokens, 0.)
        x = self.project((clean - self.mean) / self.std)
        if self.position:
            angles = positions[..., None] * (2 * math.pi) * self.frequencies
            x = x + torch.stack((angles.sin(), angles.cos()), dim=-1).flatten(-2)
        if self.plane_embedding is not None:
            x = x + self.plane_embedding[None, :, None, :]
        b, planes, slices, dim = x.shape
        flat_mask = mask.reshape(b * planes, slices)
        safe_mask = flat_mask.clone()
        safe_mask[~safe_mask.any(1), 0] = True
        x = self.attention(x.reshape(b * planes, slices, dim), src_key_padding_mask=~safe_mask)
        x = torch.where(flat_mask[..., None], x, 0.)
        vectors = x.sum(1) / flat_mask.sum(1).clamp_min(1)[:, None]
        return self.classifier(vectors.reshape(b, planes * dim))

def make_folds(ids, y):
    from sklearn.model_selection import KFold
    if len(set(ids)) != len(ids) or len(y) != len(ids) or not np.isin(y, [0, 1]).all():
        raise ValueError("Expected unique IDs and complete binary labels.")
    order = np.argsort(ids)
    folds = np.full(len(ids), -1, dtype=int)
    for fold, (train, valid) in enumerate(KFold(3, shuffle=True, random_state=42).split(order)):
        for index in (order[train], order[valid]):
            count = y[index].sum(0)
            if not ((count > 0) & (count < len(index))).all():
                raise ValueError("Each fold/label needs positives and negatives.")
        folds[order[valid]] = fold
    return folds

def metric_scores(y, logits):
    from sklearn.metrics import roc_auc_score
    probabilities = logits.sigmoid().numpy()
    aucs = roc_auc_score(y, probabilities, average=None)
    return {"macro_auc": float(np.mean(aucs)), "per_label_auc": dict(zip(LABELS, aucs.tolist())),
            "bce": nn.functional.binary_cross_entropy_with_logits(logits, torch.from_numpy(y)).item()}

def save_submission(ids, probabilities, sample_columns, target):
    probabilities = np.asarray(probabilities)
    if (not ids or len(set(ids)) != len(ids) or list(sample_columns) != ["StudyInstanceUID", *LABELS]
            or probabilities.shape != (len(ids), 12) or not np.isfinite(probabilities).all()
            or not ((probabilities >= 0) & (probabilities <= 1)).all()):
        raise ValueError("Invalid submission IDs, label order, shape or probabilities.")
    frame = pd.DataFrame(probabilities, columns=LABELS)
    frame.insert(0, "StudyInstanceUID", ids)
    target = Path(target)
    temporary = target.with_suffix(".csv.part")
    frame.to_csv(temporary, index=False)
    temporary.replace(target)
    return frame


In [ ]:
"""Report-supervised MRI experiments; shared verbatim by train and offline submit."""

import re

def make_targets(train, labels, config):
    train = train.sort_values('StudyInstanceUID').reset_index(drop=True).copy()
    if (train.StudyInstanceUID.isna().any() or not train.StudyInstanceUID.is_unique
            or labels.StudyInstanceUID.isna().any() or not labels.StudyInstanceUID.is_unique
            or not set(labels.StudyInstanceUID) <= set(train.StudyInstanceUID)):
        raise ValueError('Invalid, duplicate or foreign study IDs in label source.')
    required = [x + suffix for x in LABELS for suffix in ('', '__conf', '__verdict')]
    if not set(required) <= set(labels.columns):
        raise ValueError('Missing report label columns.')
    values = labels[[*LABELS, *[x+'__conf' for x in LABELS]]].to_numpy(dtype=np.float32)
    if not np.isfinite(values).all() or not ((values >= 0) & (values <= 1)).all():
        raise ValueError('Weak scores/confidences must be finite values in [0,1].')
    if not labels[[x+'__verdict' for x in LABELS]].isin(['YES', 'NO', 'UNK']).all().all():
        raise ValueError('Unknown report verdict.')
    observed = train[LABELS].notna()
    if (observed.any(axis=1) != observed.all(axis=1)).any():
        raise ValueError('Unexpected partially gold-labelled study.')
    gold = observed.all(axis=1).to_numpy()
    gold_y = train.loc[gold, LABELS].to_numpy(dtype=np.float32)
    if not np.isin(gold_y, [0, 1]).all():
        raise ValueError('Gold targets must be binary.')
    source = labels.set_index('StudyInstanceUID').reindex(train.StudyInstanceUID)
    y = source[LABELS].fillna(.5).to_numpy(dtype=np.float32)
    conf = source[[x+'__conf' for x in LABELS]].fillna(0).to_numpy(dtype=np.float32)
    known = source[[x+'__verdict' for x in LABELS]].isin(['YES', 'NO']).to_numpy()
    w = ((.25 + .75 * conf) * known).astype(np.float32)
    y[gold], w[gold] = gold_y, 3.
    reports = train.Report.fillna('').map(lambda s: re.sub(r'\s+', ' ', str(s)).strip().casefold())
    if reports.eq('').any():
        raise ValueError('Missing report: cannot audit report-group leakage.')
    groups = reports.map(lambda s: hashlib.sha256(s.encode()).hexdigest()).to_numpy()
    if len(set(groups[gold])) != int(gold.sum()):
        raise ValueError('Duplicate gold reports require grouped gold CV.')
    overlap = ~gold & np.isin(groups, groups[gold])
    keep = gold | (config['weak_labels'] & ~overlap & (w.sum(1) > 0))
    audit = {'gold_studies': int(gold.sum()), 'weak_studies': int((keep & ~gold).sum()),
             'weak_gold_report_overlap': int(overlap.sum()),
             'weak_without_known_targets': int((~gold & (w.sum(1) == 0)).sum()),
             'missing_source_rows': int(source.ACL.isna().sum()),
             'known_weak_targets': {name: int(((w[:, i] > 0) & keep & ~gold).sum())
                                    for i, name in enumerate(LABELS)},
             'unknown_policy': 'zero loss weight; never assume a negative',
             'teacher_gold_independence': 'not established; development CV only'}
    # Filter supervision AFTER fixing the population, keeping the MRI/cache IDs identical.
    if 'weak_min_confidence' in config:
        threshold = config['weak_min_confidence']
        if not 0 <= threshold <= 1: raise ValueError('Invalid confidence threshold.')
        removed = keep[:, None] & ~gold[:, None] & known & (conf < threshold)
        w[removed] = 0.
        verdicts = source[[x+'__verdict' for x in LABELS]].to_numpy()
        audit['confidence_filter'] = dict(threshold=threshold,
            removed_yes={k:int((removed[:,i] & (verdicts[:,i]=='YES')).sum()) for i,k in enumerate(LABELS)},
            removed_no={k:int((removed[:,i] & (verdicts[:,i]=='NO')).sum()) for i,k in enumerate(LABELS)},
            remaining_weak_weight={k:float(w[keep & ~gold,i].sum()) for i,k in enumerate(LABELS)},
            zero_supervision_weak_studies=int((keep & ~gold & (w.sum(1)==0)).sum()))
    return train.loc[keep].reset_index(drop=True), y[keep], w[keep], gold[keep], groups[keep], audit

def find_report_labels(config, input_root=Path('/kaggle/input')):
    paths = [input_root / 'rsna-knee-llm-labels/report_labels_v2.csv',
             input_root / 'datasets/pilkwang/rsna-knee-llm-labels/report_labels_v2.csv']
    found = [p for p in paths if p.is_file()]
    if len(found) != 1 or file_hash(found[0]) != config['labels_sha256']:
        raise ValueError('Attach the pinned Pilkwang report label dataset; CSV hash must match.')
    return found[0]

def find_training_run(experiment_id, input_root=Path('/kaggle/input')):
    slug = 'rsna-knee-' + experiment_id.replace('_', '-') + '-train'
    paths = [input_root / slug / 'run', input_root / 'notebooks/choiyoungwhan' / slug / 'run',
             input_root / 'kernels/choiyoungwhan' / slug / 'run']
    found = [p for p in paths if (p / 'bundle/assets.json').is_file()]
    if len(found) != 1:
        raise ValueError(f'Attach exactly the pinned training output of {experiment_id}.')
    meta = json.loads((found[0] / 'bundle/assets.json').read_text())
    if meta['experiment']['id'] != experiment_id:
        raise ValueError('Dependency experiment mismatch.')
    return found[0]

def weighted_bce(logits, targets, weights):
    loss = torch.nn.functional.binary_cross_entropy_with_logits(logits.float(), targets.float(), reduction='none')
    return (loss * weights).sum() / weights.sum().clamp_min(1e-8)

def per_label_weighted_bce(logits, targets, weights):
    loss = torch.nn.functional.binary_cross_entropy_with_logits(logits.float(), targets.float(), reduction='none')
    denominator = weights.sum(0)
    valid = denominator > 0
    values = (loss * weights).sum(0) / denominator.clamp_min(1e-8)
    return (values * valid).sum() / valid.sum().clamp_min(1)

def validate_weak_bundle(path, config):
    path = Path(path).resolve()
    manifest = json.loads((path / 'assets.json').read_text())
    if manifest['experiment'] != config or manifest['labels'] != LABELS:
        raise ValueError('Wrong weak experiment bundle.')
    required = ({f'members/{eid}/assets.json' for eid in config['train_dependencies']}
                if config.get('ensemble') else {'encoder.ts', 'model.pt'})
    files = manifest['files']
    wheels = {p.relative_to(path).as_posix() for p in (path / 'wheels').glob('*.whl')}
    if (not required <= files.keys() or len(wheels) != 2 or not wheels <= files.keys()
            or not any(x.startswith('wheels/pydicom-3.0.2-') for x in wheels)
            or not any(x.startswith('wheels/python_gdcm-3.2.6-') for x in wheels)):
        raise ValueError('Missing weights or offline DICOM dependencies.')
    for name, digest in files.items():
        target = (path / name).resolve()
        if not target.is_relative_to(path) or not target.is_file() or file_hash(target) != digest:
            raise ValueError(f'Invalid bundle artifact: {name}')
    if config.get('ensemble'):
        for eid in config['train_dependencies']:
            child = (path / 'members' / eid).resolve()
            if not child.is_relative_to(path): raise ValueError('Invalid member path.')
            child_config = json.loads((child / 'assets.json').read_text())['experiment']
            if child_config['id'] != eid: raise ValueError('Wrong ensemble member.')
            validate_weak_bundle(child, child_config)
    return manifest


In [ ]:
"""Slice tokens with explicit coordinates; identical training and inference path."""

def spatial_order(folder):
    import pydicom
    paths = sorted(folder.glob('*.dcm'))
    if not paths:
        raise FileNotFoundError(folder)
    headers = [pydicom.dcmread(p, stop_before_pixels=True, specific_tags=[
        'ImagePositionPatient', 'ImageOrientationPatient', 'InstanceNumber']) for p in paths]
    physical = all(hasattr(h, 'ImagePositionPatient') and hasattr(h, 'ImageOrientationPatient') for h in headers)
    if physical:
        orientation = np.asarray(headers[0].ImageOrientationPatient, dtype=float)
        if any(not np.allclose(h.ImageOrientationPatient, orientation, atol=1e-4) for h in headers):
            raise ValueError('Inconsistent orientations inside one series.')
        normal = np.cross(orientation[:3], orientation[3:])
        if not .99 < np.linalg.norm(normal) < 1.01:
            raise ValueError('Invalid DICOM orientation.')
        values = np.array([np.dot(h.ImagePositionPatient, normal) for h in headers])
    elif all(hasattr(h, 'InstanceNumber') for h in headers):
        values = np.array([float(h.InstanceNumber) for h in headers])
    else:
        raise ValueError('Missing physical and instance coordinates.')
    if not np.isfinite(values).all():
        raise ValueError('Non-finite DICOM coordinates.')
    order = np.argsort(values, kind='stable')
    coords = np.zeros((len(paths), 3), dtype=np.float32)
    coords[:, 0] = np.arange(len(paths))
    coords[:, 1] = values[order] - values[order][0] if physical else 0.
    coords[:, 2] = float(physical)
    return [paths[i] for i in order], coords

def selected_windows(length, config):
    if length < 1 or config['neighbors'] not in (3, 5, 10) or config['centers'] < 1:
        raise ValueError('Invalid slice selection.')
    start, end = config.get('center_range', [.25, .75])
    if not 0 <= start < end <= 1: raise ValueError('Invalid center range.')
    centers = np.unique(np.rint(np.linspace(start, end, config['centers']) * (length-1)).astype(int))
    if config['representation'] == 'triplet':
        if config['neighbors'] != 3:
            raise ValueError('Triplet encoder requires three channels.')
        return [tuple(np.clip([c-1,c,c+1], 0, length-1)) for c in centers]
    if config['representation'] != 'single':
        raise ValueError('Unknown representation.')
    left = (config['neighbors'] - 1) // 2
    indices = {int(np.clip(c+j, 0, length-1)) for c in centers
               for j in range(-left, config['neighbors']-left)}
    return [(i,) for i in sorted(indices)]

def spatial_views(uid, series, image_root, transform, config, audit=None):
    count = config['centers'] * (config['neighbors'] if config['representation'] == 'single' else 1)
    images = torch.zeros(6, count, 3, config['crop'], config['crop'])
    mask = torch.zeros(6, count, dtype=torch.bool)
    coords = torch.zeros(6, count, 3)
    rows = series.loc[series.StudyInstanceUID == uid]
    for slot, (plane, fs) in enumerate((p, f) for p in PLANES for f in (True, False)):
        choices = rows.loc[(rows.Anatomical_Plane == plane) & (rows.Fat_Suppression == fs)]
        choices = choices.sort_values(['Fluid_Sensitive', 'SeriesInstanceUID'], ascending=[False, True])
        if choices.empty:
            continue
        paths, positions = spatial_order(image_root / uid / choices.iloc[0].SeriesInstanceUID)
        decoded = {}
        windows=selected_windows(len(paths),config)
        if audit is not None:
            indices=sorted({i for window in windows for i in window})
            audit.append(dict(StudyInstanceUID=uid,slot=slot,plane=plane,fat_suppression=fs,
                original_slices=len(paths),unique_selected_slices=len(indices),tokens=len(windows),
                selected_index_min=min(indices),selected_index_max=max(indices),valid_fraction=len(windows)/count))
        for j, window in enumerate(windows):
            for index in window:
                if index not in decoded:
                    decoded[index] = image_tensor(paths[index], lambda x:x)[0]
            image = torch.stack([decoded[i] for i in window]) if len(window) == 3 else decoded[window[0]][None].repeat(3,1,1)
            images[slot, j] = transform(image)
            coords[slot, j] = torch.from_numpy(positions[window[len(window)//2]])
            mask[slot, j] = True
    if not mask.any():
        raise ValueError(f'No supported images for {uid}')
    return images, mask, coords

def spatial_signature(config):
    signature = {k: config[k] for k in ('representation', 'neighbors', 'centers', 'crop',
                                       'resize', 'interpolation', 'pipeline_sha256')}
    signature['pipeline_sha256'] = config.get('feature_pipeline_sha256', config['pipeline_sha256'])
    # Legacy configs used this exact range; keep their serialized cache signature valid.
    if list(config.get('center_range', [.25,.75])) != [.25,.75]:
        signature['center_range'] = list(config['center_range'])
    return signature

@torch.inference_mode()
def spatial_tokens(uid, series, image_root, encoder, transform, config, device, audit=None):
    images, mask, coords = spatial_views(uid, series, image_root, transform, config, audit=audit)
    output = torch.cat([encoder(x.to(device)).cpu().float() for x in images[mask].split(16)])
    if output.shape != (int(mask.sum()), config['feature_dim']) or not torch.isfinite(output).all():
        raise ValueError('Invalid spatial features.')
    tokens = torch.zeros(*mask.shape, config['feature_dim'], dtype=torch.float16)
    tokens[mask] = output.half()
    return tokens, mask, coords

def rotary(x, position):
    dim = x.shape[-1]
    if dim % 2:
        raise ValueError('Rotary head dimension must be even.')
    frequencies = torch.exp(torch.arange(0, dim, 2, device=x.device, dtype=torch.float32) * (-math.log(10000.) / dim))
    angle = position.float()[:,None,:,None] * frequencies
    a, b = x.float()[...,0::2], x.float()[...,1::2]
    return torch.stack((a*angle.cos()-b*angle.sin(), a*angle.sin()+b*angle.cos()), dim=-1).flatten(-2).to(x.dtype)

class SliceBlock(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.norm1, self.norm2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.qkv, self.out = nn.Linear(dim, 3*dim), nn.Linear(dim, dim)
        self.ffn = nn.Sequential(nn.Linear(dim,2*dim),nn.GELU(),nn.Linear(2*dim,dim))

    def forward(self, x, mask, position, use_rope):
        b, t, d = x.shape
        q,k,v = self.qkv(self.norm1(x)).reshape(b,t,3,self.heads,d//self.heads).permute(2,0,3,1,4).unbind(0)
        if use_rope:
            q,k = rotary(q,position),rotary(k,position)
        scores = (q.float() @ k.float().transpose(-1,-2)) / math.sqrt(d//self.heads)
        scores = scores.masked_fill(~mask[:,None,None,:], -1e4).softmax(-1)
        context = (scores.to(v.dtype) @ v).transpose(1,2).reshape(b,t,d)
        x = x + self.out(context)
        return (x + self.ffn(self.norm2(x))) * mask[...,None]

class SpatialHead(nn.Module):
    def __init__(self, config, mean, std):
        super().__init__()
        self.config = config
        d = config['hidden_dim']
        if d % config['heads'] or (d//config['heads']) % 2:
            raise ValueError('Invalid attention dimensions.')
        self.register_buffer('mean', mean.float().clone())
        self.register_buffer('std', std.float().clone())
        self.project = nn.Linear(config['feature_dim'], d)
        self.pool_score = nn.Sequential(nn.Linear(d,64),nn.Tanh(),nn.Linear(64,12))
        self.class_weight = nn.Parameter(torch.empty(12,6*d))
        self.class_bias = nn.Parameter(torch.zeros(12))
        nn.init.normal_(self.class_weight,std=.01)
        self.coord_kind = nn.Embedding(2,d)
        nn.init.zeros_(self.coord_kind.weight)
        if config['sequence']:
            self.slice_block = SliceBlock(d,config['heads'])
        if config['plane_embedding']:
            self.plane = nn.Embedding(3,d)
            self.protocol = nn.Embedding(2,d)
        mode = config['fusion']
        if mode == 'mlp':
            self.mlp = nn.Sequential(nn.Linear(6*d,d),nn.GELU(),nn.Linear(d,1))
        elif mode == 'gate':
            self.gate = nn.Linear(d,1)
            self.gate_weight = nn.Parameter(torch.randn(12,d)*.01)
        elif mode == 'cross':
            self.queries = nn.Parameter(torch.randn(12,d)*.02)
            self.memory_norm, self.query_norm = nn.LayerNorm(d),nn.LayerNorm(d)
            self.cross = nn.MultiheadAttention(d,config['heads'],batch_first=True,dropout=0.)
            self.read_norm = nn.LayerNorm(d)
            self.read_ffn = nn.Sequential(nn.Linear(d,2*d),nn.GELU(),nn.Linear(2*d,d))
            self.read_weight = nn.Parameter(torch.randn(12,d)*.01)
        elif mode != 'concat':
            raise ValueError('Unsupported fusion.')

    def forward(self, tokens, mask, coords):
        if tokens.ndim != 4 or tokens.shape[1] != 6 or mask.shape != tokens.shape[:3] or coords.shape != (*mask.shape,3):
            raise ValueError('Invalid token/coordinate shape.')
        if not mask.flatten(1).any(1).all():
            raise ValueError('Each study requires a valid image.')
        coords = torch.where(mask[...,None],coords,0.)
        clean = torch.where(mask[...,None],tokens.float(),self.mean)
        if not torch.isfinite(coords).all() or not torch.isfinite(clean).all():
            raise ValueError('Non-finite valid token or coordinate.')
        x = self.project((clean-self.mean)/self.std)
        x = x + self.coord_kind((coords[...,2]>.5).long())
        b,s,t,d = x.shape
        if self.config['plane_embedding']:
            slot=torch.arange(6,device=x.device)
            x=x+(self.plane(slot//2)+self.protocol(slot%2))[None,:,None,:]
        pos=coords[...,0]
        if self.config['position']=='rope_mm':
            pos=torch.where(coords[...,2]>.5,coords[...,1]/self.config['mm_unit'],pos)
        if self.config['position']=='additive':
            freq=torch.exp(torch.arange(0,d,2,device=x.device)*(-math.log(10000.)/d))
            angles=pos[...,None]*freq
            x=x+torch.stack((angles.sin(),angles.cos()),-1).flatten(-2)
        x=x*mask[...,None]
        if self.config['sequence']:
            flat=x.reshape(b*s,t,d); flat_mask=mask.reshape(b*s,t); active=flat_mask.any(1)
            result=torch.zeros_like(flat)
            result[active]=self.slice_block(flat[active],flat_mask[active],pos.reshape(b*s,t)[active],self.config['position'].startswith('rope'))
            x=result.reshape(b,s,t,d)
        if self.config['fusion']=='cross':
            memory=self.memory_norm(x.flatten(1,2)); query=self.queries[None].expand(b,-1,-1)
            read,_=self.cross(self.query_norm(query),memory,memory,key_padding_mask=~mask.flatten(1),need_weights=False)
            read=query+read; read=read+self.read_ffn(self.read_norm(read))
            return (read*self.read_weight).sum(-1)+self.class_bias
        attention=self.pool_score(x).masked_fill(~mask[...,None],-1e4).softmax(2)*mask[...,None]
        pooled=torch.einsum('bstc,bstd->bscd',attention,x).permute(0,2,1,3)
        if self.config['fusion']=='mlp':
            return self.mlp(pooled.flatten(2)).squeeze(-1)+self.class_bias
        if self.config['fusion']=='gate':
            weight=self.gate(pooled).squeeze(-1).masked_fill(~mask.any(2)[:,None,:],-1e4).softmax(-1)
            return ((pooled*weight[...,None]).sum(2)*self.gate_weight).sum(-1)+self.class_bias
        return (pooled.flatten(2)*self.class_weight).sum(-1)+self.class_bias


In [ ]:
def build_encoder(pretrained=True):
    from torchvision.models import Swin_T_Weights, swin_t
    model = swin_t(weights=Swin_T_Weights.IMAGENET1K_V1 if pretrained else None)
    model.head = nn.Identity()
    return model


In [ ]:
"""Training entry point; embedded after the core and one backbone constructor."""

import importlib.metadata

import platform

import subprocess

import sys

import time

def export_encoder(encoder, config, target):
    torch.backends.mha.set_fastpath_enabled(False)
    encoder = encoder.cpu().eval().requires_grad_(False)
    size = config["crop"]
    example = torch.zeros(1, 3, size, size)
    with torch.no_grad():
        # Fixed H/W is part of the bundle contract; batch sizes are independently checked below.
        traced = torch.jit.trace(encoder, example, check_trace=False)
        for batch_size in range(1, config["slices"] + 1):
            probe = torch.rand(batch_size, 3, size, size)
            actual, expected = traced(probe), encoder(probe)
            print(f"Encoder export batch={batch_size}: max_abs_error={(actual - expected).abs().max().item():.8g}", flush=True)
            # Float32 eager/JIT execution can round differently near zero, as observed in DINOv2.
            torch.testing.assert_close(actual, expected, rtol=1e-4, atol=1e-4)
    traced.save(str(target))
    loaded = torch.jit.load(str(target), map_location="cpu").eval()
    with torch.no_grad():
        torch.testing.assert_close(loaded(probe), encoder(probe), rtol=1e-4, atol=1e-4)
    return loaded


In [ ]:
"""Train report-supervised ablations and save fully offline inference bundles."""

import shutil

import subprocess

import sys

import time

def write_weak_metrics(config, runs, out):
    metrics = {'experiment': config, 'runs': runs,
               'macro_auc_mean': float(np.mean([r['macro_auc'] for r in runs])),
               'macro_auc_seed_std': float(np.std([r['macro_auc'] for r in runs])),
               'fold_auc_mean': float(np.mean([f['macro_auc'] for r in runs for f in r['folds']])),
               'bce_mean': float(np.mean([r['bce'] for r in runs])),
               'note': 'Gold58 development OOF; public teacher independence is not established. Seed SD is not a CI.'}
    (out / 'metrics.json').write_text(json.dumps(metrics, indent=2))
    return metrics

def freeze_swin_except_last_stage(encoder):
    encoder.eval().requires_grad_(False)
    encoder.features[-1].requires_grad_(True)
    encoder.norm.requires_grad_(True)
    return [p for p in encoder.parameters() if p.requires_grad]

def write_weak_manifest(config, bundle, table_count, gold_count, extra=None):
    manifest = {'experiment': config, 'labels': LABELS, 'python': list(sys.version_info[:2]),
                'torch': torch.__version__, 'final_training_studies': table_count,
                'gold_training_studies': gold_count, 'weak_training_studies': table_count-gold_count,
                'files': {p.relative_to(bundle).as_posix(): file_hash(p) for p in sorted(bundle.rglob('*'))
                          if p.is_file() and p != bundle / 'assets.json'}, **(extra or {})}
    (bundle / 'assets.json').write_text(json.dumps(manifest, indent=2))
    return manifest

def train_weak_ensemble(config, data, out):
    members = [find_training_run(eid) for eid in config['train_dependencies']]
    frames = [pd.read_csv(path / 'oof_predictions.csv').sort_values(['seed','StudyInstanceUID']).reset_index(drop=True)
              for path in members]
    pd.testing.assert_frame_equal(frames[0][['StudyInstanceUID','seed','fold']], frames[1][['StudyInstanceUID','seed','fold']])
    frame = frames[0].copy()
    frame[LABELS] = (frames[0][LABELS].to_numpy() + frames[1][LABELS].to_numpy()) / 2
    truth = pd.read_csv(data / 'train.csv').set_index('StudyInstanceUID')
    runs = []
    for seed, rows in frame.groupby('seed'):
        y = truth.loc[rows.StudyInstanceUID, LABELS].to_numpy(dtype=np.float32)
        logits = torch.logit(torch.from_numpy(rows[LABELS].to_numpy(dtype=np.float32)).clamp(1e-7, 1-1e-7))
        scores = [{'fold': fold, **metric_scores(y[rows.fold.to_numpy() == fold], logits[rows.fold.to_numpy() == fold])}
                  for fold in range(3)]
        runs.append({'seed': int(seed), **metric_scores(y, logits), 'folds': scores})
    frame.to_csv(out / 'oof_predictions.csv', index=False)
    shutil.copy2(members[0] / 'folds.csv', out / 'folds.csv')
    metrics = write_weak_metrics(config, runs, out)
    bundle = out / 'bundle'
    metadata = []
    for eid, path in zip(config['train_dependencies'], members):
        meta = json.loads((path / 'bundle/assets.json').read_text())
        validate_weak_bundle(path / 'bundle', meta['experiment'])
        metadata.append(meta)
        shutil.copytree(path / 'bundle', bundle / 'members' / eid)
    shutil.copytree(members[0] / 'bundle/wheels', bundle / 'wheels')
    if any(m['final_training_studies'] != config['expected_training_studies'] for m in metadata):
        raise ValueError('Ensemble members use different training populations.')
    write_weak_manifest(config, bundle, config['expected_training_studies'], 58,
                        {'members': config['train_dependencies'], 'mixing': 'fixed probability mean 0.5/0.5'})
    print(json.dumps(metrics, indent=2), flush=True)


In [ ]:
"""Train coordinate-aware heads and export offline bundles."""

import shutil

import subprocess

import sys

import time

def feature_statistics(tokens, mask, indices):
    total=torch.zeros(tokens.shape[-1],dtype=torch.float64)
    squares=torch.zeros_like(total); count=0
    for batch in torch.as_tensor(indices,dtype=torch.long).split(32):
        values=tokens[batch][mask[batch]].double()
        if not torch.isfinite(values).all(): raise ValueError('Invalid training features.')
        total+=values.sum(0); squares+=(values*values).sum(0); count+=len(values)
    if count<2: raise ValueError('Insufficient training features.')
    mean=total/count
    std=((squares-total.square()/count)/(count-1)).clamp_min(0).sqrt().clamp_min(.01)
    return mean.float(),std.float()

def fit_spatial_head(tokens,mask,coords,y,weights,train_index,seed,config):
    device='cuda' if torch.cuda.is_available() else 'cpu'
    torch.manual_seed(seed)
    index=torch.as_tensor(train_index,dtype=torch.long)
    head=SpatialHead(config,*feature_statistics(tokens,mask,index)).to(device)
    optimizer=torch.optim.AdamW(head.parameters(),lr=config['lr'],weight_decay=config['weight_decay'])
    history=[]
    for epoch in range(1,config['epochs']+1):
        head.train(); loss_sum=weight_sum=0.; steps=0
        order=index[torch.randperm(len(index))]
        for batch in order.split(config['batch_size']):
            if weights[batch].sum().item() == 0: continue
            optimizer.zero_grad(set_to_none=True)
            logits=head(tokens[batch].to(device),mask[batch].to(device),coords[batch].to(device))
            loss_fn=per_label_weighted_bce if config.get('loss_reduction')=='per_label' else weighted_bce
            loss=loss_fn(logits,y[batch].to(device),weights[batch].to(device))
            if not torch.isfinite(loss): raise ValueError('Non-finite head loss.')
            loss.backward(); torch.nn.utils.clip_grad_norm_(head.parameters(),5.); optimizer.step()
            w=1. if config.get('loss_reduction')=='per_label' else weights[batch].sum().item()
            loss_sum+=loss.item()*w; weight_sum+=w; steps+=1
        if not steps: raise ValueError('No supervised head batches.')
        history.append(dict(epoch=epoch,train_loss=loss_sum/weight_sum,optimizer_steps=steps))
    return head.eval(),history

@torch.inference_mode()
def spatial_predict(head,tokens,mask,coords,index):
    device=next(head.parameters()).device
    return torch.cat([head(tokens[b].to(device),mask[b].to(device),coords[b].to(device)).cpu()
                      for b in torch.as_tensor(index,dtype=torch.long).split(32)])

def evaluate_spatial_cv(tokens,mask,coords,y,weights,gold,ids,config,out):
    gold_index=np.flatnonzero(gold); weak_index=np.flatnonzero(~gold)
    gold_ids=np.asarray(ids)[gold].tolist(); gold_y=y[gold_index].numpy()
    folds=make_folds(gold_ids,gold_y)
    pd.DataFrame(dict(StudyInstanceUID=gold_ids,fold=folds)).to_csv(out/'folds.csv',index=False)
    runs=[]; frames=[]; histories=[]
    for seed in SEEDS:
        oof=torch.full((len(gold_index),12),float('nan')); scores=[]
        for fold in range(3):
            train_index=np.concatenate((weak_index,gold_index[folds!=fold])); valid=gold_index[folds==fold]
            head,history=fit_spatial_head(tokens,mask,coords,y,weights,train_index,seed,config)
            logits=spatial_predict(head,tokens,mask,coords,valid); oof[folds==fold]=logits
            scores.append(dict(fold=fold,**metric_scores(gold_y[folds==fold],logits)))
            histories.extend(dict(seed=seed,fold=fold,**row) for row in history)
            print(f"seed={seed} fold={fold} AUC={scores[-1]['macro_auc']:.5f}",flush=True)
            del head
        if not torch.isfinite(oof).all(): raise ValueError('Incomplete OOF predictions.')
        runs.append(dict(seed=seed,**metric_scores(gold_y,oof),folds=scores))
        frame=pd.DataFrame(oof.sigmoid().numpy(),columns=LABELS)
        frame.insert(0,'fold',folds); frame.insert(0,'seed',seed); frame.insert(0,'StudyInstanceUID',gold_ids)
        frames.append(frame)
    pd.concat(frames,ignore_index=True).to_csv(out/'oof_predictions.csv',index=False)
    pd.DataFrame(histories).to_csv(out/'history.csv',index=False)
    return write_weak_metrics(config,runs,out)

def weak_finetune_split(gold, groups):
    weak=np.flatnonzero(~gold)
    valid=weak[np.array([int(g[:8],16)%10==0 for g in groups[weak]])]
    train_index=np.setdiff1d(weak,valid)
    if not len(train_index) or not len(valid) or gold[train_index].any() or gold[valid].any(): raise ValueError('Invalid FT split.')
    return train_index,valid

def finetune_spatial(encoder,cache,table,y,weights,gold,groups,series,data,config,out):
    train_index,valid=weak_finetune_split(gold,groups)
    torch.manual_seed(42)
    warmup_epochs=config.get('head_warmup_epochs',0)
    if warmup_epochs:
        # Fresh head: the S11 final head has seen all gold and must NEVER initialize FT.
        encoder.eval().requires_grad_(False)
        head,warmup=fit_spatial_head(cache['tokens'],cache['mask'],cache['coords'],y,weights,
            train_index,42,dict(config,epochs=warmup_epochs,lr=.001))
        pd.DataFrame(warmup).to_csv(out/'warmup_history.csv',index=False)
    else:
        head=SpatialHead(config,*feature_statistics(cache['tokens'],cache['mask'],train_index)).cuda()
    encoder=encoder.cuda(); parameters=freeze_swin_except_last_stage(encoder)
    head=head.cuda()
    optimizer=torch.optim.AdamW([dict(params=parameters,lr=1e-5),dict(params=head.parameters(),lr=1e-4)],weight_decay=.01)
    scaler=torch.amp.GradScaler('cuda'); transform=make_transform(config)
    deadline=time.monotonic()+7*3600; best=float('inf'); history=[]; best_epoch=None

    def predict(index):
        if time.monotonic()>deadline: raise TimeoutError('Spatial FT exceeded 7h budget.')
        images,mask,coords=spatial_views(table.StudyInstanceUID.iloc[index],series,data/'train_series',transform,config)
        with torch.autocast('cuda',dtype=torch.float16):
            features=torch.cat([encoder(chunk.cuda()) for chunk in images[mask].split(8)])
            device_mask=mask.cuda(); tokens=features.new_zeros(*mask.shape,config['feature_dim'])
            tokens[device_mask]=features
            return head(tokens[None],device_mask[None],coords[None].cuda())

    for epoch in range(1,config['backbone_epochs']+1):
        encoder.eval(); encoder.features[-1].train(); head.train()
        order=np.random.default_rng(42+epoch).permutation(train_index); total=0.
        for start in range(0,len(order),8):
            batch=order[start:start+8]; optimizer.zero_grad(set_to_none=True)
            for index in batch:
                loss=weighted_bce(predict(index),y[index:index+1].cuda(),weights[index:index+1].cuda())
                if not torch.isfinite(loss): raise ValueError('Non-finite FT loss.')
                scaler.scale(loss/len(batch)).backward(); total+=loss.item()
            scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_([*parameters,*head.parameters()],5.)
            scaler.step(optimizer); scaler.update()
            if start%200==0: print(f'FT epoch={epoch} studies={start}/{len(order)}',flush=True)
        encoder.eval(); head.eval(); value=denominator=0.
        with torch.inference_mode():
            for index in valid:
                w=weights[index:index+1].sum().item()
                value+=weighted_bce(predict(index),y[index:index+1].cuda(),weights[index:index+1].cuda()).item()*w
                denominator+=w
        score=value/denominator
        if not np.isfinite(score): raise ValueError('Invalid weak validation loss.')
        history.append(dict(epoch=epoch,train_loss=total/len(order),weak_validation_bce=score))
        if score<best:
            best,best_epoch=score,epoch
            torch.save({k:v.detach().cpu() for k,v in encoder.state_dict().items()},out/'backbone_best.pt')
    encoder.load_state_dict(torch.load(out/'backbone_best.pt',map_location='cpu',weights_only=True))
    pd.DataFrame(history).to_csv(out/'backbone_history.csv',index=False)
    (out/'backbone_audit.json').write_text(json.dumps(dict(training_weak_studies=len(train_index),
        validation_weak_studies=len(valid),gold_studies_used_for_backbone=0,best_epoch=best_epoch,
        validation_bce=best,head_warmup_epochs=warmup_epochs,warmup_gold_studies=0,
        trainable_parameters=sum(p.numel() for p in parameters))),encoding='utf-8')
    return encoder.cpu().eval().requires_grad_(False)

def compatible_spatial_cache(cache,config,encoder_hash,ids):
    return (cache.get('view_signature')==spatial_signature(config) and cache.get('encoder_sha256')==encoder_hash
            and cache.get('ids')==list(ids))

def train_spatial_experiment(config,encoder_factory,data,out):
    from tqdm.auto import tqdm
    data,out=Path(data),Path(out)
    if out.exists() and any(out.iterdir()): raise FileExistsError('Choose an empty output directory.')
    out.mkdir(parents=True,exist_ok=True)
    if config.get('ensemble'): return train_weak_ensemble(config,data,out)
    if not torch.cuda.is_available(): raise RuntimeError('A Kaggle GPU is required.')
    torch.set_num_threads(1); torch.manual_seed(42); torch.backends.mha.set_fastpath_enabled(False)
    table,y,w,gold,groups,audit=make_targets(pd.read_csv(data/'train.csv'),pd.read_csv(find_report_labels(config)),config)
    if len(table)!=4350 or int(gold.sum())!=58: raise ValueError('Training population changed.')
    y,w=torch.from_numpy(y),torch.from_numpy(w)
    audit.update(source='pilkwang/rsna-knee-llm-labels/versions/1',sha256=config['labels_sha256'])
    (out/'label_audit.json').write_text(json.dumps(audit,indent=2),encoding='utf-8')
    series=pd.read_csv(data/'train_series.csv')
    if not series.SeriesInstanceUID.is_unique or not set(table.StudyInstanceUID)<=set(series.StudyInstanceUID):
        raise ValueError('Invalid MRI series metadata.')
    bundle=out/'bundle'; bundle.mkdir(); previous=None; cache=None
    if config['train_dependencies']:
        previous=find_training_run(config['train_dependencies'][0])
        meta=json.loads((previous/'bundle/assets.json').read_text())
        validate_weak_bundle(previous/'bundle',meta['experiment'])
        if file_hash(previous/'tokens.pt')!=meta['tokens_sha256']: raise ValueError('Corrupt cached tokens.')
        cache=torch.load(previous/'tokens.pt',map_location='cpu',weights_only=True)
        shutil.copy2(previous/'bundle/encoder.ts',bundle/'encoder.ts')
        encoder=torch.jit.load(str(bundle/'encoder.ts'),map_location='cpu').eval()
    else:
        encoder=export_encoder(encoder_factory(),dict(config,slices=16),bundle/'encoder.ts')
    ids=table.StudyInstanceUID.tolist()
    backbone_started=time.monotonic()
    if config['finetune']:
        if not compatible_spatial_cache(cache,config,file_hash(bundle/'encoder.ts'),ids): raise ValueError('FT requires matching cache.')
        eager=encoder_factory(pretrained=False); eager.load_state_dict(encoder.state_dict(),strict=True)
        eager=finetune_spatial(eager,cache,table,y,w,gold,groups,series,data,config,out)
        encoder=export_encoder(eager,dict(config,slices=16),bundle/'encoder.ts'); del eager
        cache=None
    backbone_seconds=time.monotonic()-backbone_started if config['finetune'] else 0.
    encoder_hash=file_hash(bundle/'encoder.ts'); started=time.monotonic(); reused=False
    torch.cuda.reset_peak_memory_stats()
    if cache is not None and compatible_spatial_cache(cache,config,encoder_hash,ids):
        tokens,mask,coords=(cache[k] for k in ('tokens','mask','coords')); reused=True
    else:
        cache=None; encoder=encoder.cuda().eval(); transform=make_transform(config); rows=[]; selection_audit=[]
        for uid in tqdm(ids,desc=config['id']+' MRI features'):
            if time.monotonic()-started>7*3600: raise TimeoutError('Feature extraction exceeded 7h budget.')
            rows.append(spatial_tokens(uid,series,data/'train_series',encoder,transform,config,'cuda',audit=selection_audit))
        pd.DataFrame(selection_audit).to_csv(out/'selection_audit.csv',index=False)
        tokens,mask,coords=[torch.stack([row[i] for row in rows]) for i in range(3)]
        del rows
    if not torch.isfinite(tokens).all() or not torch.isfinite(coords).all(): raise ValueError('Invalid feature cache.')
    torch.save(dict(ids=ids,tokens=tokens,mask=mask,coords=coords,view_signature=spatial_signature(config),encoder_sha256=encoder_hash),out/'tokens.pt')
    timing=dict(feature_seconds=time.monotonic()-started,backbone_seconds=backbone_seconds,images=int(mask.sum()),cached_studies=len(ids) if reused else 0,
                peak_cuda_allocated_mb=torch.cuda.max_memory_allocated()/1024**2,gpu=torch.cuda.get_device_name(0),
                physical_coordinate_tokens=int(((coords[...,2]>.5)&mask).sum()),index_fallback_tokens=int(((coords[...,2]<=.5)&mask).sum()))
    del cache,encoder; torch.cuda.empty_cache()
    metrics=evaluate_spatial_cv(tokens,mask,coords,y,w,gold,ids,config,out)
    head,history=fit_spatial_head(tokens,mask,coords,y,w,np.arange(len(table)),42,config)
    torch.save(dict(state={k:v.detach().cpu() for k,v in head.state_dict().items()},experiment=config,labels=LABELS),bundle/'model.pt')
    pd.DataFrame(history).to_csv(out/'final_fit_history.csv',index=False)
    if previous: shutil.copytree(previous/'bundle/wheels',bundle/'wheels')
    else:
        (bundle/'wheels').mkdir()
        subprocess.check_call([sys.executable,'-m','pip','download','-q','--only-binary=:all:','--no-deps','--dest',str(bundle/'wheels'),
                               'pydicom==3.0.2','python-gdcm==3.2.6'])
    write_weak_manifest(config,bundle,len(table),int(gold.sum()),dict(head_parameters=sum(p.numel() for p in head.parameters()),
        tokens_sha256=file_hash(out/'tokens.pt'),folds_sha256=file_hash(out/'folds.csv'),label_audit=audit))
    (out/'timings.json').write_text(json.dumps(timing,indent=2),encoding='utf-8')
    print(json.dumps(dict(macro_auc=metrics['macro_auc_mean'],timing=timing),indent=2),flush=True)


In [ ]:
"""Two-factor position ablation: categorical plane x continuous slice depth.

Kept separate so an already running spatial/follow-up queue keeps its source hash.
This head retains S11's per-series RoPE and class-query cross attention.
"""

class PlaneDepthHead(SpatialHead):
    def __init__(self, config, mean, std):
        if (config['fusion'] != 'cross' or not config['sequence'] or
                not config['plane_embedding'] or config['position'] != 'rope_mm'):
            raise ValueError('Plane-depth ablation requires the fixed S11 architecture.')
        if config.get('plane_depth_mode') not in ('none', 'sum', 'joint'):
            raise ValueError('Unknown plane-depth encoding.')
        if not math.isfinite(config['mm_unit']) or config['mm_unit'] <= 0:
            raise ValueError('mm_unit must be positive and finite.')
        super().__init__(config, mean, std)

    def position_features(self, position):
        """(B, 6, T) -> (B, 6, T, D), with no new trainable parameters.

        방향은 0/1/2의 연속 좌표가 아닌 Sagittal/Coronal/Axial 범주이다.
        slice 위치는 series 첫 장에서의 mm 거리 / mm_unit; 없으면 정렬 index.
        sum: 방향 embedding과 깊이 sin/cos를 별도로 더한다.
        joint: 방향 embedding으로 깊이 신호를 조절하는 상호작용도 더한다.
        tanh는 조절량을 [-1, 1]로 제한한다. 실제 학습에서는 기존 plane
        embedding으로 gradient가 흐른다. 이것은 2D RoPE나 patch (x,y)가 아니다.
        """
        d = self.config['hidden_dim']
        frequencies = torch.exp(torch.arange(0, d, 2, device=position.device,
                                            dtype=torch.float32) * (-math.log(10000.) / d))
        angles = position.float()[..., None] * frequencies
        depth = torch.stack((angles.sin(), angles.cos()), -1).flatten(-2)
        mode = self.config['plane_depth_mode']
        if mode == 'none':
            return torch.zeros_like(depth)
        if mode == 'sum':
            return depth
        plane = self.plane(torch.arange(6, device=position.device) // 2)
        return depth + torch.tanh(plane)[None, :, None, :] * depth

    def forward(self, tokens, mask, coords):
        # S11 cross branch reproduced here to leave active historical code untouched.
        # A regression test requires exact S11 outputs/gradients in the 'none' control.
        if tokens.ndim != 4 or tokens.shape[1] != 6 or mask.shape != tokens.shape[:3] or coords.shape != (*mask.shape, 3):
            raise ValueError('Invalid token/coordinate shape.')
        if not mask.flatten(1).any(1).all():
            raise ValueError('Each study requires a valid image.')
        coords = torch.where(mask[..., None], coords, 0.)
        clean = torch.where(mask[..., None], tokens.float(), self.mean)
        if not torch.isfinite(coords).all() or not torch.isfinite(clean).all():
            raise ValueError('Non-finite valid token or coordinate.')
        x = self.project((clean - self.mean) / self.std)
        x = x + self.coord_kind((coords[..., 2] > .5).long())
        b, s, t, d = x.shape
        slot = torch.arange(6, device=x.device)
        x = x + (self.plane(slot // 2) + self.protocol(slot % 2))[None, :, None, :]
        pos = torch.where(coords[..., 2] > .5, coords[..., 1] / self.config['mm_unit'], coords[..., 0])
        # NEW: absolute depth is visible to the cross-view class queries as well.
        # Existing RoPE below still supplies relative depth inside each series.
        if self.config['plane_depth_mode'] != 'none':
            x = x + self.position_features(pos)
        x = x * mask[..., None]
        flat = x.reshape(b * s, t, d)
        flat_mask = mask.reshape(b * s, t)
        active = flat_mask.any(1)
        result = torch.zeros_like(flat)
        result[active] = self.slice_block(flat[active], flat_mask[active],
                                         pos.reshape(b * s, t)[active], True)
        x = result.reshape(b, s, t, d)
        memory = self.memory_norm(x.flatten(1, 2))
        query = self.queries[None].expand(b, -1, -1)
        read, _ = self.cross(self.query_norm(query), memory, memory,
                             key_padding_mask=~mask.flatten(1), need_weights=False)
        read = query + read
        read = read + self.read_ffn(self.read_norm(read))
        return (read * self.read_weight).sum(-1) + self.class_bias


In [ ]:
"""Small head/ensemble additions; historical training sources stay unchanged."""

import json

from pathlib import Path

import numpy as np

import pandas as pd

import torch

from torch import nn

def blend_weights(config):
    count = len(config['train_dependencies'])
    weights = np.asarray(config.get('blend_weights', [1 / count] * count if count else []), dtype=float)
    if (not count or weights.shape != (count,) or not np.isfinite(weights).all()
            or (weights < 0).any() or not np.isclose(weights.sum(), 1., rtol=0, atol=1e-8)):
        raise ValueError('Blend weights must be finite, nonnegative, and sum to one.')
    return weights

def blend_oof(frames, weights):
    weights = blend_weights(dict(train_dependencies=list(range(len(frames))), blend_weights=weights))
    keys = ['StudyInstanceUID', 'seed', 'fold']
    aligned = []
    for frame in frames:
        if frame.columns.tolist() != [*keys, *LABELS] or frame.duplicated(keys[:2]).any() or not len(frame):
            raise ValueError('Invalid OOF columns, empty input, or duplicate study/seed.')
        frame = frame.sort_values(keys[:2]).reset_index(drop=True)
        if frame[keys].isna().any().any() or (aligned and not frame[keys].equals(aligned[0][keys])):
            raise ValueError('OOF study/seed/fold alignment differs.')
        values = frame[LABELS].to_numpy(float)
        if not np.isfinite(values).all() or not ((values >= 0) & (values <= 1)).all():
            raise ValueError('Invalid OOF probabilities.')
        aligned.append(frame)
    result = aligned[0].copy()
    result[LABELS] = sum(w * frame[LABELS].to_numpy(float) for w, frame in zip(weights, aligned))
    return result

def drop_slices(mask, probability):
    if (mask.ndim != 3 or mask.dtype != torch.bool or not 0 <= probability < 1
            or not mask.flatten(1).any(1).all()):
        raise ValueError('Invalid slice dropout mask/probability.')
    if probability == 0:
        return mask
    keep = mask & (torch.rand(mask.shape, device=mask.device) >= probability)
    empty = mask.any(-1) & ~keep.any(-1)
    # Retain the first valid token only when random dropping emptied a present slot.
    rescue = torch.zeros_like(mask).scatter_(-1, mask.long().argmax(-1, keepdim=True), True)
    return keep | (rescue & empty[..., None])

class DropoutSpatialHead(SpatialHead):
    def forward(self, tokens, mask, coords):
        if self.training and self.config.get('slice_dropout', 0):
            mask = drop_slices(mask, self.config['slice_dropout'])
        return super().forward(tokens, mask, coords)

class SeedMeanHead(nn.Module):
    def __init__(self, heads):
        super().__init__()
        if not heads:
            raise ValueError('A seed ensemble needs trained heads.')
        self.heads = nn.ModuleList(heads)

    def forward(self, tokens, mask, coords):
        probabilities = torch.stack([head(tokens, mask, coords).sigmoid() for head in self.heads]).mean(0)
        return torch.logit(probabilities.clamp(1e-7, 1 - 1e-7))

def restore_head(config, state):
    offsets = config.get('head_seed_offsets', [0])
    if len(offsets) > 1:
        head = SeedMeanHead([DropoutSpatialHead(config, state[f'heads.{i}.mean'], state[f'heads.{i}.std'])
                             for i in range(len(offsets))])
    else:
        cls = PlaneDepthHead if 'plane_depth_mode' in config else DropoutSpatialHead
        head = cls(config, state['mean'], state['std'])
    head.load_state_dict(state, strict=True)
    return head


In [ ]:
_base_fit = fit_spatial_head
"""Reuse CV and fold-only statistics; only head fitting and blend weights differ."""

import json

from pathlib import Path

import numpy as np

import pandas as pd

import torch

def fit_campaign_head(tokens, mask, coords, y, weights, train_index, seed, config):
    offsets = config.get('head_seed_offsets', [0])
    if not offsets or len(set(offsets)) != len(offsets) or any(type(x) != int or x < 0 for x in offsets):
        raise ValueError('Seed offsets must be distinct nonnegative integers.')
    heads, history = [], []
    for offset in offsets:
        head, rows = _base_fit(tokens, mask, coords, y, weights, train_index, seed + offset, config)
        heads.append(head)
        history.extend(dict(row, member_seed=seed + offset) for row in rows)
    return (heads[0] if len(heads) == 1 else SeedMeanHead(heads).eval()), history

def train_campaign_experiment(config, encoder_factory, data, out):
    data, out = Path(data), Path(out)
    if config.get('ensemble'):
        if out.exists() and any(out.iterdir()):
            raise FileExistsError('Choose an empty output directory.')
        out.mkdir(parents=True, exist_ok=True)
        # Reuse the existing bundle-copy/hash/population checks before replacing the equal blend.
        train_weak_ensemble(config, data, out)
        frames = [pd.read_csv(find_training_run(eid) / 'oof_predictions.csv') for eid in config['train_dependencies']]
        frame = blend_oof(frames, blend_weights(config))
        truth = pd.read_csv(data / 'train.csv').set_index('StudyInstanceUID')
        runs = []
        for seed, rows in frame.groupby('seed'):
            y = truth.loc[rows.StudyInstanceUID, LABELS].to_numpy(np.float32)
            logits = torch.logit(torch.from_numpy(rows[LABELS].to_numpy(np.float32)).clamp(1e-7, 1 - 1e-7))
            scores = [dict(fold=fold, **metric_scores(y[rows.fold == fold], logits[rows.fold.to_numpy() == fold])) for fold in range(3)]
            runs.append(dict(seed=int(seed), **metric_scores(y, logits), folds=scores))
        frame.to_csv(out / 'oof_predictions.csv', index=False)
        metrics = write_weak_metrics(config, runs, out)
        path = out / 'bundle/assets.json'; manifest = json.loads(path.read_text())
        manifest['mixing'] = dict(method='fixed_probability', weights=blend_weights(config).tolist())
        path.write_text(json.dumps(manifest, indent=2))
        return metrics
    namespace = train_spatial_experiment.__globals__
    previous = {name: namespace[name] for name in ('fit_spatial_head', 'SpatialHead')}
    try:
        namespace.update(fit_spatial_head=fit_campaign_head, SpatialHead=DropoutSpatialHead)
        return train_spatial_experiment(config, encoder_factory, data, out)
    finally:
        namespace.update(previous)


In [ ]:
train_campaign_experiment(EXPERIMENT,build_encoder,find_data(),Path('/kaggle/working/run'))
